In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas_ta as ta


In [2]:
ticker = yf.Ticker("AAPL")
df = ticker.history(period="1mo", interval="1d")

display(df.head())

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-03-18 00:00:00-04:00,252.630005,254.940002,249.000000,249.940002,35757900,0.0,0.0
2026-03-19 00:00:00-04:00,249.399994,251.830002,247.300003,248.960007,34864100,0.0,0.0
2026-03-20 00:00:00-04:00,247.979996,249.199997,246.000000,247.990005,88331100,0.0,0.0
2026-03-23 00:00:00-04:00,253.970001,254.600006,250.279999,251.490005,40546100,0.0,0.0
2026-03-24 00:00:00-04:00,250.350006,254.830002,249.550003,251.639999,45152300,0.0,0.0


In [3]:
df_intraday = yf.download(tickers=["AAPL", "MSFT"], period="5d", interval="15m")

display(df_intraday['Close'].tail())

[*********************100%***********************]  2 of 2 completed


Ticker,AAPL,MSFT
Datetime,,
2026-04-17 18:45:00+00:00,270.130005,421.670013
2026-04-17 19:00:00+00:00,269.630005,421.614990
2026-04-17 19:15:00+00:00,269.695007,422.644989
2026-04-17 19:30:00+00:00,270.480011,422.980011
2026-04-17 19:45:00+00:00,270.184998,422.600006


In [4]:
fig = go.Figure(data=[go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='AAPL Price'
)])

# 3. Customize layout for a dashboard feel
fig.update_layout(
    title='Apple (AAPL) - Live Interactive Chart',
    yaxis_title='Price (USD)',
    xaxis_title='Date',
    xaxis_rangeslider_visible=False, # Hides the default bottom slider for a cleaner look
    template='plotly_dark',          # Dark mode theme
    height=600
)

# 4. Display
fig.show()

In [12]:
def fetch_and_prepare_data(ticker, period="1y", interval="1d"):
    df = yf.download(ticker, period=period, interval=interval)
    
    # Flatten MultiIndex if it exists
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Optional: Clean column names (makes them lowercase for easier access)
    df.columns = [str(c).lower() for c in df.columns]
    
    # Now pandas-ta will find 'close' automatically
    df.ta.macd(append=True)
    df.ta.rsi(length=14, append=True)
    df.ta.sma(length=20, append=True)
    df.ta.sma(length=50, append=True)

    df.dropna(inplace=True)
    
    return df

In [13]:
df_ready = fetch_and_prepare_data("AAPL")
display(df_ready.tail())

[*********************100%***********************]  1 of 1 completed


,close,high,low,open,volume,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,RSI_14,SMA_20,SMA_50
Date,,,,,,,,,,,
2026-04-13,259.200012,260.179993,256.660004,259.730011,36234700,0.079093,1.367308,-1.288215,53.658153,253.739001,260.865333
2026-04-14,258.829987,261.929993,257.190002,259.250000,48370700,0.237154,1.220295,-0.983141,53.067359,254.039500,260.857184
2026-04-15,266.429993,266.559998,257.809998,258.160004,49913500,0.964557,1.558158,-0.593601,62.258788,254.649500,260.790632
2026-04-16,263.399994,267.160004,261.269989,266.799988,43323100,1.281758,1.500288,-0.218529,57.429775,255.322499,260.674070
2026-04-17,270.230011,272.299988,266.720001,266.959991,61314800,2.060516,1.823236,0.237280,64.175150,256.385999,260.554041


In [16]:
def plot_technical_analysis(df):
    # Create a grid: 3 rows (Price, RSI, MACD)
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.05, 
                        row_heights=[0.5, 0.2, 0.3])

    # Row 1: Candlesticks + SMAs
    fig.add_trace(go.Candlestick(x=df.index, open=df['open'], high=df['high'], 
                                 low=df['low'], close=df['close'], name='price'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], name='SMA 20', line=dict(width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_50'], name='SMA 50', line=dict(width=1)), row=1, col=1)

    # Row 2: RSI
    fig.add_trace(go.Scatter(x=df.index, y=df['RSI_14'], name='RSI', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

    # Row 3: MACD
    fig.add_trace(go.Bar(x=df.index, y=df['MACDh_12_26_9'], name='Histogram'), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['MACD_12_26_9'], name='MACD'), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['MACDs_12_26_9'], name='Signal'), row=3, col=1)

    fig.update_layout(template='plotly_dark', height=800, showlegend=False, xaxis_rangeslider_visible=False)
    fig.show()

In [17]:
plot_technical_analysis(df_ready)